# Credit Risk Classification — version 2

This version follows the class workflow in a readable order: understand the data, check quality, split without leakage, preprocess categorical and numeric variables, fit candidate models, predict risk, evaluate the operating policy, and translate the result into a business recommendation.

**Business question:** which loan applications should receive enhanced human review when the review team can examine at most 20% of applications?

The positive class is `bad/default-risk`: `1` means the application is treated as bad credit risk. The model estimates risk; it does not automatically approve or reject anyone.

## 1. Load the project data

We use the immutable raw file stored in the assignment project. The root-finding helper makes the notebook work when Jupyter is launched from the project root or from the `notebooks` folder.

In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'data_contract.py').exists() and (candidate / 'data' / 'raw').exists():
            return candidate
    raise FileNotFoundError('Could not locate the assignment project root')

ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_contract import load_raw_data, make_xy, contract_summary

RAW_PATH = ROOT / 'data' / 'raw' / 'SouthGermanCredit.asc'
frame = load_raw_data(RAW_PATH)
X, y = make_xy(frame)
print(f'Project root: {ROOT}')
print(f'Raw data: {RAW_PATH}')
display(frame.head())

## 2. Understand the dataset before modelling

The source has 1,000 historical applications, 20 predictors, and one target. `kredit == 0` is mapped to the positive class `bad_default_risk == 1`; this makes the class we care about explicit.

In a real acquisition-screening model, this is the same principle: first define exactly what one row represents, what moment the prediction is made, and what outcome the model is trying to flag.

In [ ]:
summary = contract_summary(frame)
summary_table = pd.DataFrame({
    'check': ['rows', 'columns', 'missing values', 'duplicate rows', 'bad/default-risk cases', 'good/performing cases'],
    'value': [
        summary['rows'],
        summary['columns'],
        summary['missing_values'],
        summary['duplicate_rows'],
        int(y.sum()),
        int((y == 0).sum()),
    ]
})
display(summary_table)
display(pd.DataFrame({'raw_value': frame['kredit'].value_counts().sort_index(), 'meaning': ['bad/default-risk', 'good/performing']}))

In [ ]:
feature_dictionary = {
    'laufkont': 'existing checking-account status',
    'laufzeit': 'loan duration in months',
    'moral': 'credit history',
    'verw': 'loan purpose',
    'hoehe': 'credit amount',
    'sparkont': 'savings account or bonds',
    'beszeit': 'employment duration',
    'rate': 'installment rate as percentage of disposable income',
    'famges': 'personal status and sex code',
    'buerge': 'other debtors or guarantors',
    'wohnzeit': 'years at present residence',
    'verm': 'property status',
    'alter': 'age in years',
    'weitkred': 'other installment plans',
    'wohn': 'housing status',
    'bishkred': 'number of existing credits at this bank',
    'beruf': 'job category',
    'pers': 'number of people financially liable for',
    'telef': 'telephone status',
    'gastarb': 'foreign-worker code',
    'kredit': 'target: historical credit-risk outcome'
}
columns = pd.DataFrame({
    'column': frame.columns,
    'dtype': [str(frame[c].dtype) for c in frame.columns],
    'role': ['target' if c == 'kredit' else 'predictor' for c in frame.columns],
    'meaning': [feature_dictionary[c] for c in frame.columns]
})
display(columns)

## 3. Data-quality decisions

The validation contract finds no null values or duplicate rows in this source file. Therefore, we do not delete anything merely because a cleaning recipe says to do so. Duplicates could otherwise give repeated applications extra weight; nulls would require a documented imputation rule.

We also do not remove every statistical outlier automatically. A large loan amount may be unusual but still real. Removing it without a business reason could erase the very cases the model must handle.

In [ ]:
quality = pd.DataFrame({
    'column': frame.columns,
    'missing_values': frame.isna().sum().values,
    'unique_values': frame.nunique().values,
    'min': [frame[c].min() for c in frame.columns],
    'max': [frame[c].max() for c in frame.columns],
})
display(quality)
print('Rows duplicated:', int(frame.duplicated().sum()))

## 4. Train/test split: protect the test set

The split happens before fitting preprocessing. The development set is used to learn model parameters and compare candidates; the frozen test set is held back for one final confirmation. `stratify=y` keeps the bad-risk proportion similar in both parts.

This prevents data leakage: the model must not learn the test-set mean, category frequencies, scaling parameters, or target pattern before the final evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 77
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
split_table = pd.DataFrame({
    'split': ['development', 'frozen test'],
    'rows': [len(y_dev), len(y_test)],
    'bad/default-risk': [int(y_dev.sum()), int(y_test.sum())],
    'bad-risk rate': [y_dev.mean(), y_test.mean()]
})
display(split_table)

## 5. Encoding and scaling inside the model pipeline

The class notebooks teach one-hot encoding for nominal categories and scaling for numeric variables. Here both are placed inside a scikit-learn `Pipeline`, so the transformations are fitted only on the data supplied to each training fold.

That is safer than manually encoding the full dataset first. For example, `wohn` (housing) is categorical; a value such as 2 is not automatically twice as much housing as 1.

In [ ]:
from src.models import make_candidate_models

candidate_models = make_candidate_models(random_state=RANDOM_STATE)
logistic_pipeline = candidate_models['logistic_regression']
print(logistic_pipeline)
print('Pipeline steps:', list(logistic_pipeline.named_steps))

## 6. Compare candidate models on development data

`fit()` means the model learns from the development data. We compare a majority baseline, logistic regression, a constrained decision tree, and a random forest. The model comparison uses five-fold out-of-fold predictions inside the development set; the frozen test set is not used to choose the winner.

Because the business can review only 20%, the important question is not simply ‘which model has the highest accuracy?’ It is ‘which model ranks the riskiest applications most usefully within the review capacity?’

In [ ]:
oof = pd.read_csv(ROOT / 'artifacts' / 'tables' / 'wave-4-oof-threshold-table-all-models.csv')
policy_rows = oof[oof['policy'].eq('recommended_capacity_constrained')].copy()
policy_rows = policy_rows.sort_values('net_screening_value', ascending=False)
display(policy_rows[['model', 'policy', 'threshold', 'accuracy', 'precision', 'recall', 'f1', 'monthly_reviews', 'net_screening_value']])
policy_manifest = json.loads((ROOT / 'artifacts' / 'manifests' / 'wave-4-policy.json').read_text(encoding='utf-8'))
selected_model_name = policy_manifest['selected_model']
print('Selected model:', selected_model_name)

## 7. `fit()` and `predict_proba()` on the frozen test

Now the selected pipeline is fitted on the development set. `predict_proba()` returns a risk score for each previously unseen test application. For a test row, a score of `0.73` means the model ranks it as having a relatively high estimated probability of the positive class—bad/default-risk. It is not proof that default will happen.

`predict()` applies a conventional threshold, usually 0.50. The operating policy here is different: rank all scores and send exactly the highest 20% to human review.

In [ ]:
from src.evaluation import binary_metrics
from src.threshold_policy import metrics_at_capacity, BUSINESS_ASSUMPTIONS

selected_model = make_candidate_models(random_state=RANDOM_STATE)[selected_model_name]
selected_model.fit(X_dev, y_dev)
test_risk_score = selected_model.predict_proba(X_test)[:, 1]
test_prediction_at_050 = selected_model.predict(X_test)

print('First five estimated bad-risk probabilities:', np.round(test_risk_score[:5], 3))
print('First five threshold-0.50 predictions:', test_prediction_at_050[:5])
print('Manual-review capacity:', BUSINESS_ASSUMPTIONS['max_monthly_reviews'], 'of', BUSINESS_ASSUMPTIONS['monthly_applications'])

## 8. Evaluate: classification metrics and the business policy

Accuracy is the share of all classifications that are correct. Precision asks: among the applications flagged for review, how many are actually bad-risk in this historical label? Recall asks: among all bad-risk applications, how many did the queue capture?

For this case, capacity and expected screening value matter too. A high-recall policy that sends 80% of applications to a team that can inspect only 20% is not operationally valid.

In [ ]:
metrics_050 = binary_metrics(y_test, test_risk_score, threshold=0.50)
metrics_top20 = metrics_at_capacity(
    y_test, test_risk_score,
    BUSINESS_ASSUMPTIONS['max_monthly_reviews'] / BUSINESS_ASSUMPTIONS['monthly_applications']
)
metric_table = pd.DataFrame([
    {'policy': 'threshold = 0.50', **{k: metrics_050[k] for k in ['accuracy', 'precision', 'recall', 'f1', 'review_rate']}},
    {'policy': 'ranked top 20%', **{k: metrics_top20[k] for k in ['accuracy', 'precision', 'recall', 'f1', 'review_rate']}},
])
display(metric_table)
display(pd.DataFrame([metrics_top20])[['tn', 'fp', 'fn', 'tp', 'monthly_reviews', 'net_screening_value', 'cost_per_expected_default_prevented']])

In [ ]:
confusion = pd.DataFrame(
    [[metrics_top20['tn'], metrics_top20['fp']], [metrics_top20['fn'], metrics_top20['tp']]],
    index=['actual good/performing', 'actual bad/default-risk'],
    columns=['not selected for enhanced review', 'selected for enhanced review']
)
display(confusion)
print('Interpretation: TP are bad-risk cases selected for review; FN are bad-risk cases missed by the queue.')

## 9. Business interpretation and sensitivity

The financial numbers are scenario estimates from the assignment, not observed outcomes in this historical dataset. Review effectiveness, loss given default, review cost, and abandonment must be measured in a real deployment. Cambiar estas suposiciones cambia el euro estimate, but it does not change the model's classification metrics.

In [ ]:
sensitivity = pd.read_csv(ROOT / 'artifacts' / 'tables' / 'wave-5-sensitivity-test.csv')
display(sensitivity[['scenario', 'net_screening_value', 'cost_per_expected_default_prevented', 'monthly_reviews']])

## 10. Governance checks and limitations

Subgroup error rates matter when variables may be sensitive or act as proxies. Removing a protected attribute alone does not make a model fair: other correlated variables can still carry similar information, and the historical labels may themselves reflect unequal processes.

The South German Credit data is historical and old; its label is a credit-risk proxy, not a directly observed future 12-month default. The model estimates risk, and an enhanced review may or may not prevent a loss. No causal claim is justified.

In [ ]:
audit_path = ROOT / 'artifacts' / 'tables' / 'wave-9-subgroup-audit.csv'
if audit_path.exists():
    display(pd.read_csv(audit_path))
else:
    print('Governance audit table has not been generated yet:', audit_path)

## 11. Recommendation

Use the selected model as a ranking and prioritisation tool: send the highest-scoring 20% to enhanced human review, respecting the hard capacity limit. Do not automatically approve or reject applications. Applications outside the queue continue through the ordinary human credit process; they are not automatically declared safe.

The result is useful as a classroom demonstration of supervised classification and decision policy design. Before any real use, the organisation would need current data, a directly defined outcome, calibration, temporal validation, subgroup review, drift monitoring, and evidence that enhanced review actually changes outcomes.

## Quick check

1. What is the positive class? **Bad/default-risk.**
2. What does `fit()` do? **It learns preprocessing parameters and model parameters from development data.**
3. What does `predict_proba()` produce? **An estimated score/probability for each class; we use the bad-risk score for ranking.**
4. Why is the test set frozen? **To estimate performance on unseen data after choices are finished.**
5. Why not use accuracy alone? **A capacity-constrained review team needs a useful ranked queue, not merely the most common label predicted for everyone.**